In [ ]:
# %%capture
# %pip install -q -U google-genai
# %pip install google-cloud-aiplatform

In [ ]:
import ast
import time
import os

import pandas as pd
import google.generativeai as genai

# Configure Gemini API - Thêm API key của bạn vào đây
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

generation_config = {
    "temperature": 0.2,
    "top_p": 0.95,
    "top_k": 40,
}

# Safety settings
safety_settings = [
    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
]

# Initialize Gemini 2.5 Flash model
model = genai.GenerativeModel(
    model_name="gemini-2.5-flash",  
    generation_config=generation_config,
    safety_settings=safety_settings
)

input_csv = "full_data.csv"
output_csv = "summarized_content_data.csv"

In [ ]:
def generate_summary(title, content, model):
    prompt = f"""
Tóm tắt chi tiết, cụ thể bài báo sau thành 1 đoạn văn tiếng Việt. 
Giữ nguyên các thông tin quan trọng (tên người/tổ chức/công ty, số liệu (%, số tiền, số lượng), địa điểm, mốc thời gian) trong nội dung.

Tiêu đề: {title}

Nội dung:
{content}

Chỉ trả lời phần tóm tắt, không thêm tiêu đề hay giải thích.
"""
    try:
        response = model.generate_content(prompt)
        return (response.text or "").strip()
    except Exception as e:
        raise Exception(f"Lỗi khi tạo tóm tắt: {str(e)}")


def preprocess_data(datapath):
    """Đọc và xử lý dữ liệu từ CSV"""
    df = pd.read_csv(datapath)
    return df


def run_batch(input_csv, output_csv, model, sample_size=None):
    """Xử lý batch tóm tắt nội dung cho tất cả các bài báo"""
    print(f"Đang đọc dữ liệu từ {input_csv}...")
    df = preprocess_data(input_csv)
    print(f"Tổng số bài báo trong file: {len(df)}")
    
    # Nếu có sample_size, chỉ lấy random sample
    if sample_size and sample_size < len(df):
        df = df.sample(n=sample_size, random_state=42).reset_index(drop=True)
        print(f"Chạy test với {sample_size} samples ngẫu nhiên")

    all_summaries = []
    failed_rows = []

    for idx, row in df.iterrows():
        title = str(row['title'])
        content = str(row['content'])
        
        try:
            summary_text = generate_summary(title, content, model)
            all_summaries.append(summary_text)
            
            if (idx + 1) % 10 == 0:
                print(f"Đã xử lý {idx + 1}/{len(df)} bài báo...")
            
            # Rate limiting
            time.sleep(0.5)
            
        except Exception as e:
            print(f"Lỗi tại bài {idx}: {str(e)}")
            all_summaries.append("")
            failed_rows.append({'index': idx, 'title': title, 'reason': str(e)})

    df['summary'] = all_summaries

    # Keep only necessary columns
    cols_to_keep = ['url', 'title', 'time', 'content', 'summary']
    cols_to_keep = [c for c in cols_to_keep if c in df.columns]
    df = df[cols_to_keep]

    print(f"Đang lưu kết quả vào {output_csv}...")
    df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print("Hoàn thành!")
    
    # Create failed_df from failed_rows
    failed_df = pd.DataFrame(failed_rows) if failed_rows else pd.DataFrame()
    
    return df, failed_df

In [ ]:
# Test với 10 samples ngẫu nhiên
try:
    result_df, failed_df = run_batch(input_csv, output_csv, model, sample_size=10)
    print(f"\n✅ Test hoàn thành! Kết quả đã lưu vào {output_csv}")
    print(f"Số bài thành công: {len(result_df)}")
    if not failed_df.empty:
        print(f"Số bài thất bại: {len(failed_df)}")
except Exception as e:
    print(f"❌ Lỗi: {e}")

In [ ]:
if not failed_df.empty:
    failed_df.to_csv("failed_context_summarization.csv", index=False, encoding='utf-8-sig')

In [ ]:
from transformers import AutoTokenizer

# Load PhoBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)

# Đọc file kết quả
df = pd.read_csv(output_csv)

df['content_token_count'] = df['content'].apply(lambda x: len(tokenizer.tokenize(str(x))))

# Tính số lượng token cho mỗi summary
df['summary_token_count'] = df['summary'].apply(lambda x: len(tokenizer.tokenize(str(x))))

# Thống kê mô tả số lượng token
print(df[['content_token_count', 'summary_token_count']].describe())